# cerberus-neuro - environment smoke test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/00_environment_smoke.ipynb)

Run this **first** in any Colab session to verify the runtime is wired up correctly. It checks:

1. GPU available (or graceful CPU fallback)
2. PyTorch sees the GPU
3. `cerberus_neuro` package installs from GitHub and imports
4. Hugging Face Hub login works (via Colab Secret `HF_TOKEN`)
5. Google Drive mounts (optional; for persistent caching)

If all pass, the runtime is good to go for training and analysis notebooks.

First-time setup: see `docs/SETUP.md` in the repo.

## 1. GPU check

In [ ]:
!nvidia-smi || echo "No GPU allocated; runtime is CPU-only. Switch via Runtime > Change runtime type if needed."

## 2. Install cerberus-neuro from GitHub

Always pulls the latest `main`. Includes the `[training]` extra to bring in PyTorch + torchvision (Colab usually has these pre-installed; this ensures versions are compatible).

First install ~30-60 s.

In [ ]:
!pip install -q "cerberus-neuro[training] @ git+https://github.com/PatrickJReed/cerberus-neuro.git@main"

In [ ]:
import cerberus_neuro
print(f"cerberus_neuro {cerberus_neuro.__version__} imported OK")

## 3. PyTorch + GPU

Verify torch imports and sees the GPU. v0 training assumes a single CUDA device; CPU fallback works for sanity testing only.

In [ ]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU. Training will not be feasible on CPU.")

## 4. Hugging Face login

Reads token from the Colab Secret `HF_TOKEN`. Outside Colab, falls back to the local `huggingface-cli login` cache.

In [ ]:
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print("HF login: OK (via Colab secret)")
except Exception as e:
    print(f"Colab secret not used ({type(e).__name__}); relying on local huggingface-cli cache")

api = HfApi()
me = api.whoami()
print(f"HF authenticated as: {me['name']}")

## 5. Drive mount (optional, Colab only)

Persistent storage at `/content/drive/MyDrive/cerberus-neuro/` survives session resets. Skip if running outside Colab.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cerberus-neuro/cache'
    ckpt_dir = '/content/drive/MyDrive/cerberus-neuro/checkpoints'
    os.makedirs(cache_dir, exist_ok=True)
    os.makedirs(ckpt_dir, exist_ok=True)
    print(f"Drive mounted; cache + checkpoint dirs ready under /content/drive/MyDrive/cerberus-neuro/")
except (ImportError, Exception) as e:
    print(f"Drive mount skipped ({type(e).__name__}): {e}")

## 6. Smoke-test summary

In [ ]:
import sys
import platform

print("cerberus-neuro smoke test summary")
print("-" * 40)
print(f"Python:         {sys.version.split()[0]}")
print(f"Platform:       {platform.platform()}")
print(f"cerberus_neuro: {cerberus_neuro.__version__}")
print(f"torch:          {torch.__version__}")
print(f"CUDA:           {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"HF user:        {me['name']}")
print("")
print("If all sections above ran without errors, the runtime is good.")